# 📊 Data Visualization Model Lab — Complete Practical Solutions

**Student:** Arul Agnes
**Course:** M.E. CSE — Data Visualization
**Date:** 2025
**Dataset:** `cse_student_performance_1200.csv` — 1,200 rows × 20 columns (SYNTHETIC data)

> Every one of the 10 lab questions is solved TWICE below: a **simple** readable version and a comprehensive **advanced** version, with interpretation after every chart.

## 📁 Setup and Dataset Loading

In [ ]:
# Install any missing packages (quiet = no wall of text)
!pip install umap-learn statsmodels --quiet

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
print("Libraries loaded. pandas", pd.__version__, "| seaborn", sns.__version__)

In [ ]:
# Load the dataset - try every plausible path (Kaggle input, local, downloaded)
csv_path = None
candidates = [
    "/kaggle/input/cse-student-performance-1200/cse_student_performance_1200.csv",
    "/kaggle/input/cse-student-performance/cse_student_performance_1200.csv",
    "/kaggle/input/data-v/cse_student_performance_1200.csv",
    "cse_student_performance_1200.csv",
    "../datasets/cse_student_performance_1200.csv",
    "datasets/cse_student_performance_1200.csv",
]
for path in candidates:
    if os.path.exists(path):
        csv_path = path
        break

# last resort: download the public dataset from Kaggle Hub
if csv_path is None:
    try:
        import kagglehub
        folder = kagglehub.dataset_download("arulmariaagnes/cse-student-performance-1200")
        csv_path = os.path.join(folder, "cse_student_performance_1200.csv")
    except Exception as e:
        print("kagglehub fallback failed:", e)

if csv_path is None:
    print("CSV not found! Use 'Add Input' on Kaggle and attach")
    print("dataset: arulmariaagnes/cse-student-performance-1200")
else:
    df = pd.read_csv(csv_path)
    print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from:")
    print(" ", csv_path)
    print(df.head())

## Question 1: Bar Chart, Histogram and Box Plot

**Concepts:** comparing categories (bar), distribution shape (histogram), spread + outliers (box plot).

In [ ]:
# --- Q1: three basic charts ---
sec_mean = df.groupby("section")["final_exam_marks"].mean()

# (1) bar chart: average marks per section, with value labels
plt.figure(figsize=(8, 5))
bars = plt.bar(sec_mean.index, sec_mean.values, color="#2196F3", edgecolor="black")
for bar, v in zip(bars, sec_mean.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{v:.1f}", ha="center", fontweight="bold")
plt.title("Average Final Exam Marks by Section")
plt.xlabel("Section"); plt.ylabel("Average Marks"); plt.ylim(0, 100)
plt.tight_layout(); plt.savefig("q01_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()

# (2) histogram of all final marks with a mean line
plt.figure(figsize=(8, 5))
plt.hist(df["final_exam_marks"], bins=20, color="#4CAF50", edgecolor="black", alpha=0.85)
m = df["final_exam_marks"].mean()
plt.axvline(m, color="red", linestyle="--", linewidth=2, label=f"Mean = {m:.1f}")
plt.title("Distribution of Final Exam Marks")
plt.xlabel("Marks"); plt.ylabel("Number of Students"); plt.legend()
plt.tight_layout(); plt.savefig("q01_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

# (3) box plot: spread of marks inside each section
plt.figure(figsize=(8, 5))
sns.boxplot(x="section", y="final_exam_marks", data=df, order=list("ABCD"),
            hue="section", palette="Set2", legend=False)
plt.title("Final Exam Marks Distribution by Section")
plt.xlabel("Section"); plt.ylabel("Marks")
plt.tight_layout(); plt.savefig("q01_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

spread = df.groupby("section")["final_exam_marks"].std()
print(f"Section {sec_mean.idxmax()} has the highest average ({sec_mean.max():.1f})")
print(f"Marks range from {df['final_exam_marks'].min():.1f} to {df['final_exam_marks'].max():.1f}")
print(f"Section {spread.idxmax()} varies the most (std = {spread.max():.1f})")

**Interpretation:** bar charts compare category averages, the histogram shows the roughly
bell-shaped spread of marks around the mean, and the box plot shows every section has a similar
median with a handful of low outliers.

### 🔬 Advanced version — `full_version/q01_advanced_categorical_numerical.py`

Comprehensive categorical analysis: grouped bars (section x department),
overlaid histograms, box plots coloured by gender, a count plot, plus a
full statistical summary table per group.


In [ ]:
"""
Question 1: Categorical and Numerical Visualization (Comprehensive)
Advanced Version — Data Visualization Lab
Author: Arul Agnes
Date: 2025

Comprehensive categorical analysis: grouped bars (section x department),
overlaid histograms, box plots coloured by gender, a count plot, plus a
full statistical summary table per group.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
PALETTE = ["#2196F3", "#4CAF50", "#FF9800"]


def group_summary(data, by):
    """Return counts, mean, median and std of final marks for every group."""
    out = data.groupby(by)["final_exam_marks"].agg(
        count="count", mean="mean", median="median", std="std")
    return out.round(2)


# ---------------------------------------------------------------------------
# FIGURE 1 - GROUPED BAR: average marks by section AND department
# ---------------------------------------------------------------------------
sec_dept = df.pivot_table(index="section", columns="department",
                          values="final_exam_marks", aggfunc=np.mean)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sec_dept.plot.bar(ax=axes[0], color=PALETTE, edgecolor="black", width=0.75)
axes[0].set_title("Average Final Marks: Section x Department", fontsize=13)
axes[0].set_xlabel("Section")
axes[0].set_ylabel("Average Marks")
axes[0].set_ylim(0, 100)
axes[0].legend(title="Department")
axes[0].tick_params(axis="x", rotation=0)
for container in axes[0].containers:                 # value labels on every bar
    axes[0].bar_label(container, fmt="%.1f", fontsize=8, padding=2)

# count plot: how many students per department (context for the averages)
dept_count = df["department"].value_counts().reindex(["CSE", "CSE-AI", "CSE-DS"])
bars = axes[1].bar(dept_count.index, dept_count.values,
                   color=PALETTE, edgecolor="black")
for bar, v in zip(bars, dept_count.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
                 str(v), ha="center", fontweight="bold")
axes[1].set_title("Number of Students per Department", fontsize=13)
axes[1].set_xlabel("Department")
axes[1].set_ylabel("Student Count")
fig.tight_layout()
fig.savefig("q01_grouped_bar.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# FIGURE 2 - STACKED/OVERLAID HISTOGRAM: marks distribution by department
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5.5))
bins = np.arange(0, 105, 5)
for dept, color in zip(["CSE", "CSE-AI", "CSE-DS"], PALETTE):
    subset = df.loc[df["department"] == dept, "final_exam_marks"]
    ax.hist(subset, bins=bins, density=True, alpha=0.55, edgecolor="black",
            label=f"{dept} (n={len(subset)})", color=color)
    ax.axvline(subset.mean(), color=color, linestyle="--", linewidth=2)
ax.set_title("Distribution of Final Marks by Department (density)", fontsize=13)
ax.set_xlabel("Final Exam Marks")
ax.set_ylabel("Density")
ax.legend(title="Department")
fig.tight_layout()
fig.savefig("q01_stacked_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# FIGURE 3 - GROUPED BOX PLOT: marks by section, coloured by gender
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.boxplot(x="section", y="final_exam_marks", hue="gender", data=df,
            order=["A", "B", "C", "D"], palette="Set2", ax=ax)
# overlay group means as diamonds
means = df.groupby(["section", "gender"])["final_exam_marks"].mean().reset_index()
sns.stripplot(x="section", y="final_exam_marks", hue="gender", data=means,
              order=["A", "B", "C", "D"], palette="Set2", marker="D",
              size=9, ax=ax, dodge=True, legend=False, color="black")
ax.set_title("Final Marks by Section, Coloured by Gender (diamond = group mean)",
             fontsize=13)
ax.set_xlabel("Section")
ax.set_ylabel("Final Exam Marks")
ax.legend(title="Gender", bbox_to_anchor=(1.01, 1), loc="upper left")
fig.tight_layout()
fig.savefig("q01_grouped_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# DETAILED STATISTICAL SUMMARY
# ---------------------------------------------------------------------------
print("=" * 72)
print("SUMMARY BY SECTION")
print(group_summary(df, "section").to_string())
print("\nSUMMARY BY DEPARTMENT")
print(group_summary(df, "department").to_string())
print("\nSUMMARY BY GENDER")
print(group_summary(df, "gender").to_string())
print("\nCROSS-TAB: section x department")
print(pd.crosstab(df["section"], df["department"]).to_string())

print("\nINTERPRETATION")
sec_means = df.groupby("section")["final_exam_marks"].mean()
print(f"- Section means differ by only {sec_means.max() - sec_means.min():.1f} "
      "marks - sections are quite similar")
print(f"- CSE has {len(df[df.department == 'CSE'])} students, so its averages "
      "are the most reliable; CSE-DS (n=100) is noisiest")
print("- Grouped bars compare two categories at once; overlaid histograms show "
      "shape; box plots expose outliers inside each group")

## Question 2: Time-Series Line Chart

**Concepts:** datetime conversion, grouping by date, trends over time.

In [ ]:
# --- Q2: average marks on each assessment date ---
df["assessment_date"] = pd.to_datetime(df["assessment_date"])
by_date = (df.groupby("assessment_date")["final_exam_marks"].mean()
             .reset_index().sort_values("assessment_date"))

plt.figure(figsize=(12, 5))
plt.plot(by_date["assessment_date"], by_date["final_exam_marks"],
         marker="o", color="#2196F3", linewidth=2)
plt.title("Average Student Performance Over Semester")
plt.xlabel("Assessment Date"); plt.ylabel("Average Final Exam Marks")
plt.xticks(rotation=45); plt.grid(alpha=0.4)
plt.tight_layout(); plt.savefig("q02_time_series.png", dpi=150, bbox_inches="tight")
plt.show()

first, last = by_date["final_exam_marks"].iloc[0], by_date["final_exam_marks"].iloc[-1]
peak = by_date.loc[by_date["final_exam_marks"].idxmax()]
valley = by_date.loc[by_date["final_exam_marks"].idxmin()]
trend = "INCREASING" if last > first + 1 else ("DECREASING" if last < first - 1 else "STABLE")
print(f"Performance trend: {trend} ({first:.1f} -> {last:.1f})")
print(f"Highest average {peak['assessment_date'].date()} = {peak['final_exam_marks']:.1f}")
print(f"Lowest average  {valley['assessment_date'].date()} = {valley['final_exam_marks']:.1f}")

**Interpretation:** the line wobbles because each date assesses a *different* group of
students — treat single-date jumps as sampling noise unless they persist over several weeks.

### 🔬 Advanced version — `full_version/q02_advanced_time_series.py`

Plots several measures over the semester on dual axes, fits a linear
trend line to performance, annotates peak/valley weeks, and checks
whether the pattern is a real trend or small-sample noise.


In [ ]:
"""
Question 2: Time Series (Multi-Series)
Advanced Version — Data Visualization Lab
Author: Arul Agnes
Date: 2025

Plots several measures over the semester on dual axes, fits a linear
trend line to performance, annotates peak/valley weeks, and checks
whether the pattern is a real trend or small-sample noise.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

df["assessment_date"] = pd.to_datetime(df["assessment_date"])
sns.set_style("whitegrid")

# ---------------------------------------------------------------------------
# Weekly aggregation (smoother than raw dates)
# ---------------------------------------------------------------------------
df["week"] = df["assessment_date"].dt.isocalendar().week.astype(int)
weekly = df.groupby("week").agg(
    date=("assessment_date", "min"),
    marks=("final_exam_marks", "mean"),
    attendance=("attendance_percentage", "mean"),
    study=("study_hours_per_week", "mean"),
    n=("final_exam_marks", "count"),
).reset_index()
print("WEEKLY AGGREGATES")
print(weekly.round(2).to_string(index=False))

# linear trend on marks: y = slope * time + intercept
time_num = np.arange(len(weekly))            # week index 0,1,2,...
slope, intercept = np.polyfit(time_num, weekly["marks"], 1)
trend = slope * time_num + intercept
trend_dir = "UPWARD" if slope > 0 else "DOWNWARD"

peak = weekly.loc[weekly["marks"].idxmax()]
valley = weekly.loc[weekly["marks"].idxmin()]

# ---------------------------------------------------------------------------
# FIGURE 1 - MULTI-SERIES with dual y-axes
# ---------------------------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(weekly["date"], weekly["marks"], marker="o", color="#2196F3",
         linewidth=2, label="Avg Final Marks (left)")
ax1.plot(weekly["date"], weekly["attendance"], marker="s", color="#4CAF50",
         linewidth=1.8, linestyle="--", label="Avg Attendance % (left)")
ax1.plot(weekly["date"], trend, color="#2196F3", linestyle=":", linewidth=2.5,
         label=f"Linear trend ({slope:+.2f} marks/week)")
ax1.set_ylabel("Marks / Attendance (%)")
ax1.set_ylim(0, 100)
ax1.set_xlabel("Week of Semester")

# right axis for study hours (different unit -> second axis is justified here)
ax2 = ax1.twinx()
ax2.plot(weekly["date"], weekly["study"], marker="^", color="#FF5722",
         linewidth=1.8, label="Avg Study Hours (right)")
ax2.set_ylabel("Study Hours per Week")
ax2.set_ylim(0, 30)

# highlight start / mid / end of semester
ax1.axvspan(weekly["date"].iloc[0], weekly["date"].iloc[3],
            alpha=0.08, color="green", label="Start of semester")
ax1.axvspan(weekly["date"].iloc[len(weekly) // 2 - 1],
            weekly["date"].iloc[len(weekly) // 2 + 1],
            alpha=0.08, color="orange", label="Mid semester")
ax1.axvspan(weekly["date"].iloc[-4], weekly["date"].iloc[-1],
            alpha=0.08, color="red", label="End of semester")

# annotate peak and valley
ax1.annotate(f"Peak {peak['marks']:.1f}", xy=(peak["date"], peak["marks"]),
             xytext=(10, 15), textcoords="offset points",
             arrowprops=dict(arrowstyle="->"), fontsize=10, fontweight="bold")
ax1.annotate(f"Valley {valley['marks']:.1f}", xy=(valley["date"], valley["marks"]),
             xytext=(10, -25), textcoords="offset points",
             arrowprops=dict(arrowstyle="->"), fontsize=10, fontweight="bold")

# proper date formatting
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha="right")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left", fontsize=9)
ax1.set_title("Semester Performance: Marks, Attendance, Study Hours and Trend",
              fontsize=14)
fig.tight_layout()
fig.savefig("q02_multi_series.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# FIGURE 2 - SEASONAL view: marks grouped by month + week box plot
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
df["month"] = df["assessment_date"].dt.strftime("%Y-%m")
sns.boxplot(x="month", y="final_exam_marks", data=df, palette="Blues", ax=axes[0])
axes[0].set_title("Marks by Month (seasonal view)", fontsize=13)
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Final Exam Marks")

# week-to-week spread shows how noisy each assessment was
sns.boxplot(x="week", y="final_exam_marks", data=df, palette="viridis", ax=axes[1])
axes[1].set_title("Marks by Week (week-to-week spread)", fontsize=13)
axes[1].set_xlabel("Week number")
axes[1].set_ylabel("")
fig.tight_layout()
fig.savefig("q02_seasonal.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# INTERPRETATION
# ---------------------------------------------------------------------------
month_means = df.groupby("month")["final_exam_marks"].mean()
print("\nINTERPRETATION")
print(f"- Trend: {trend_dir} at {slope:+.3f} marks per week "
      f"(total change over semester ~ {slope * len(weekly):+.1f} marks)")
print(f"- Peak week {int(peak['week'])} ({peak['date'].date()}) = {peak['marks']:.1f}; "
      f"valley week {int(valley['week'])} ({valley['date'].date()}) = {valley['marks']:.1f}")
print("- Monthly means:", ", ".join(f"{m}={v:.1f}" for m, v in month_means.items()))
print(f"- Sample sizes per week range {weekly['n'].min()}-{weekly['n'].max()} students; "
      "weeks with small n fluctuate more - check before calling a jump a 'trend'.")
print("- A genuine trend should persist across many weeks AND have decent sample "
      "sizes; a single wild week is usually noise.")

## Question 3: Misleading Chart and Its Fix

**Concepts:** truncated axes, colour accessibility, honest design.

In [ ]:
# --- Q3: deliberately BAD chart vs honest chart ---
dept_mean = df.groupby("department")["final_exam_marks"].mean()

# BAD: axis starts at 55 (exaggerates), rainbow colours, no labels
plt.figure(figsize=(8, 5))
plt.bar(dept_mean.index, dept_mean.values, color=["#ff0000", "#00cc00", "#0066ff"],
        edgecolor="black")
plt.ylim(55, 60); plt.yticks([]); plt.xticks([])
plt.tight_layout(); plt.savefig("q03_misleading.png", dpi=150, bbox_inches="tight")
plt.show()

# GOOD: axis from 0, colourblind-safe colour, title + labels + value labels
plt.figure(figsize=(8, 5))
bars = plt.bar(dept_mean.index, dept_mean.values, color="#2196F3", edgecolor="black")
for bar, v in zip(bars, dept_mean.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             f"{v:.1f}", ha="center", fontweight="bold")
plt.ylim(0, 100)
plt.title("Average Final Marks by Department")
plt.xlabel("Department"); plt.ylabel("Average Marks")
plt.tight_layout(); plt.savefig("q03_fixed.png", dpi=150, bbox_inches="tight")
plt.show()

print("Issue 1: truncated y-axis exaggerated tiny differences  -> Fix: axis starts at 0")
print("Issue 2: red/green is unreadable for colour-blind viewers -> Fix: single blue")
print("Issue 3: no title or labels made it uninterpretable       -> Fix: added both")
print("Issue 4: fake 3D/decoration distorts perception           -> Fix: clean 2D bars")
print(f"Real spread between departments: only {dept_mean.max()-dept_mean.min():.1f} marks")

**Interpretation:** the same numbers look like a 10× difference when the axis is truncated.
**Rule:** bar charts must start at 0.

### 🔬 Advanced version — `full_version/q03_advanced_redesign.py`

Builds a deliberately WORSE chart with 6 distinct misleading issues,
puts it side-by-side with an honest redesign, demonstrates two fixes as
mini-visualizations, and prints a report card citing perception theory.

> Note: the advanced misleading chart is saved as `q03_misleading_adv.png` so it does not overwrite the simple version's figure.


In [ ]:
"""
Question 3: Misleading Visualization Redesign (Detailed)
Advanced Version — Data Visualization Lab
Author: Arul Agnes
Date: 2025

Builds a deliberately WORSE chart with 6 distinct misleading issues,
puts it side-by-side with an honest redesign, demonstrates two fixes as
mini-visualizations, and prints a report card citing perception theory.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

dept_mean = df.groupby("department")["final_exam_marks"].mean()
depts = list(dept_mean.index)
values = list(dept_mean.values)
print("HONEST VALUES:", {d: round(v, 2) for d, v in zip(depts, values)})

# rainbow colours = pretty but meaningless and inaccessible
RAINBOW = ["#ff0000", "#ff8800", "#ffee00", "#00cc00", "#0066ff", "#8800cc"]

# ===========================================================================
# THE 6 MISLEADING ISSUES
#   1. Truncated y-axis (starts at 55) exaggerates differences
#   2. 3D-style pie chart for data that suits bars
#   3. Rainbow colour scheme (colour-blind hostile, implies fake order)
#   4. Missing title, axis labels and legend
#   5. Overcrowded overlapping labels
#   6. Dual y-axes scaled to force a fake "trend"
# ===========================================================================

# --- mini chart of issues 2,3,4,5: a broken 3D-ish pie ---
fig = plt.figure(figsize=(13, 9))
ax_pie = fig.add_subplot(2, 2, 1)
wedges, _, autotexts = ax_pie.pie(
    values, labels=None, colors=RAINBOW[:3], autopct="%1.1f%%",
    startangle=90, explode=(0.12, 0.12, 0.12), shadow=True,  # issue 2: decoration
    textprops=dict(fontsize=8))
for i, t in enumerate(autotexts):                 # issue 5: crowded labels
    t.set_position((t.get_position()[0] * 0.55, t.get_position()[1] * 0.55))
    t.set_fontsize(6)
# issue 4: NO title, NO labels, NO legend

# --- issue 1 + 6: truncated-axis bars with a fake dual axis ---
ax_bad = fig.add_subplot(2, 2, 2)
x = np.arange(len(depts))
ax_bad.bar(x, values, color=RAINBOW[:3], edgecolor="black", width=0.6)  # issue 3
ax_bad.set_ylim(55, 60)                           # issue 1: truncated axis
ax_bad.set_yticks([])                             # hides the truncation
ax_bad.set_xticks(x)
ax_bad.set_xticklabels([])                        # issue 4: no labels
ax2 = ax_bad.twinx()                              # issue 6: needless 2nd axis
noise = np.linspace(0.1, 0.9, len(depts))
ax2.plot(x, noise, color="red", linewidth=3)      # fake "trend" on dummy scale
ax2.set_ylim(0, 1)
ax2.set_yticks([])
for i, (xi, yi) in enumerate(zip(x, noise)):      # issue 5: overlapping text
    ax_bad.text(xi, 55.1, f"{values[i]:.2f} marks dept {depts[i]} avg score",
                rotation=90, fontsize=7, ha="center")

fig.suptitle("", fontsize=1)   # issue 4: intentionally no overall title
fig.tight_layout()
fig.savefig("q03_misleading_adv.png", dpi=150, bbox_inches="tight")
plt.show()

# ===========================================================================
# BEFORE / AFTER COMPARISON
# ===========================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# BEFORE (re-drawn compactly with issues 1,3,4)
axes[0, 0].bar(x, values, color=RAINBOW[:3], edgecolor="black", width=0.6)
axes[0, 0].set_ylim(55, 60)
axes[0, 0].set_yticks([])
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels([])
axes[0, 0].set_title("BEFORE: misleading", fontsize=12)

# AFTER - honest bar chart
bars = axes[0, 1].bar(depts, values, color="#2196F3", edgecolor="black")
for b, v in zip(bars, values):
    axes[0, 1].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.6,
                    f"{v:.1f}", ha="center", fontweight="bold")
axes[0, 1].set_ylim(0, 100)
axes[0, 1].set_title("AFTER: title + labels + zero-based axis", fontsize=12)
axes[0, 1].set_xlabel("Department")
axes[0, 1].set_ylabel("Average Final Marks")

# MINI-FIX 1: same bars, two axis scales side by side
axes[1, 0].bar(depts, values, color="#4CAF50", edgecolor="black")
axes[1, 0].set_ylim(55, 60)
axes[1, 0].set_title("Fix demo: axis from 55 (dishonest)", fontsize=11)
mini = axes[1, 0].inset_axes([0.55, 0.08, 0.42, 0.35])   # small inset with axis at 0
mini.bar(depts, values, color="#4CAF50", edgecolor="black")
mini.set_ylim(0, 100)
mini.set_title("axis from 0 (honest)", fontsize=8)
mini.tick_params(labelsize=7)

# MINI-FIX 2: rainbow vs single accessible colour
axes[1, 1].bar(depts, values, color=RAINBOW[:3], edgecolor="black")
axes[1, 1].set_ylim(0, 100)
axes[1, 1].set_title("Fix demo: rainbow (bad) vs single colour (good)", fontsize=11)
for i, v in enumerate(values):
    axes[1, 1].bar(i + 0.02, v, width=0.28, color="#2196F3", edgecolor="black")

fig.suptitle("Misleading Chart vs Honest Redesign", fontsize=15)
fig.tight_layout()
fig.savefig("q03_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

# ===========================================================================
# REPORT CARD
# ===========================================================================
spread_real = max(values) - min(values)
print("\nREPORT CARD")
print(f"{'#':>2} {'ISSUE':52s} {'FIX'}")
rows = [
    (1, "Truncated y-axis (55-60) makes {0:.1f} mark gaps look huge".format(spread_real),
     "Axis starts at 0; differences shown honestly"),
    (2, "3D/pie chart: angle+area are hard to compare, adds clutter",
     "Plain 2D bars: position on a common scale"),
    (3, "Rainbow colours: inaccessible & imply meaningless order",
     "Single colourblind-safe blue"),
    (4, "No title/labels/legend: chart is uninterpretable",
     "Descriptive title and axis labels added"),
    (5, "Overcrowded overlapping text hides the message",
     "Few, bold value labels only"),
    (6, "Dual y-axes rescaled to invent a fake trend",
     "One axis per unit; second axis only when units differ"),
]
for num, issue, fix in rows:
    print(f"{num:>2} {issue:52s} {fix}")

print("\nINTERPRETATION (perception theory)")
print("- Stevens' power law: perceived quantity grows as a POWER of intensity -")
print("  bar length (exponent ~1) is far more accurate than area (~0.7) or angle.")
print("- Pre-attentive features (colour, position) are processed instantly -")
print("  use them for the ONE message you want the reader to see first.")
print(f"- The true department spread is only {spread_real:.1f} marks; the misleading")
print("  chart made it look like a factor-of-10 difference.")

## Question 4: Compare Two Graphical Encodings

**Concepts:** position/length vs size encoding, Cleveland–McGill hierarchy.

In [ ]:
# --- Q4: same data, two encodings ---
sec_stats = df.groupby("section").agg(marks=("final_exam_marks", "mean"),
                                      hours=("study_hours_per_week", "mean")).reset_index()
sec_stats["num"] = range(1, 5)

# encoding 1: bars (position + length)
plt.figure(figsize=(8, 5))
plt.bar(sec_stats["section"], sec_stats["marks"], color="#2196F3", edgecolor="black")
plt.ylim(0, 100); plt.title("Encoding 1: Bar Chart (Position + Length)")
plt.xlabel("Section"); plt.ylabel("Average Marks")
plt.tight_layout(); plt.savefig("q04_encoding_1.png", dpi=150, bbox_inches="tight")
plt.show()

# encoding 2: bubbles (size)
plt.figure(figsize=(8, 5))
plt.scatter(sec_stats["num"], sec_stats["hours"], s=sec_stats["marks"] * 12,
            c="#FF5722", alpha=0.65, edgecolors="black")
for x, y, m in zip(sec_stats["num"], sec_stats["hours"], sec_stats["marks"]):
    plt.annotate(f"{m:.1f}", (x, y), ha="center", va="center", color="white",
                 fontweight="bold")
plt.xticks(sec_stats["num"], sec_stats["section"])
plt.title("Encoding 2: Bubble Chart (Size)")
plt.xlabel("Section"); plt.ylabel("Average Study Hours / Week")
plt.tight_layout(); plt.savefig("q04_encoding_2.png", dpi=150, bbox_inches="tight")
plt.show()

# redesign: two honest bar charts
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].bar(sec_stats["section"], sec_stats["marks"], color="#2196F3", edgecolor="black")
ax[0].set(title="Average Marks", xlabel="Section", ylabel="Marks", ylim=(0, 100))
ax[1].bar(sec_stats["section"], sec_stats["hours"], color="#4CAF50", edgecolor="black")
ax[1].set(title="Average Study Hours", xlabel="Section", ylabel="Hours", ylim=(0, 20))
plt.tight_layout(); plt.savefig("q04_redesigned.png", dpi=150, bbox_inches="tight")
plt.show()

print("Bar chart wins: eyes compare LENGTH on a common scale far better than AREA.")
print("Cleveland-McGill ranking: position > length > angle > area > colour.")

**Interpretation:** bubble areas double misleadingly (Stevens' power law) — a 58 vs 62
bubble pair is nearly impossible to tell apart, while two bars make the gap obvious.

### 🔬 Advanced version — `full_version/q04_advanced_encoding.py`

Compares three encodings of the same numbers - grouped bars (position +
length), bubbles (size + colour) and a heatmap (colour intensity) -
ranks them with the Cleveland & McGill hierarchy, then redesigns the
weakest encoding.


In [ ]:
"""
Question 4: Encoding Comparison (Theory-Backed)
Advanced Version — Data Visualization Lab
Author: Arul Agnes
Date: 2025

Compares three encodings of the same numbers - grouped bars (position +
length), bubbles (size + colour) and a heatmap (colour intensity) -
ranks them with the Cleveland & McGill hierarchy, then redesigns the
weakest encoding.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# the SAME numbers encoded three different ways
sec_mean = df.groupby("section")["final_exam_marks"].mean()
sections = list(sec_mean.index)
marks = list(sec_mean.values)

# ---------------------------------------------------------------------------
# ENCODINGS: (1) position+length  (2) area+colour  (3) colour intensity
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1 - grouped/simple bar: position on a common scale
axes[0].bar(sections, marks, color="#2196F3", edgecolor="black", width=0.6)
for i, v in enumerate(marks):
    axes[0].text(i, v + 0.8, f"{v:.1f}", ha="center", fontweight="bold")
axes[0].set_ylim(0, 100)
axes[0].set_title("1. Position + Length\n(most accurate)", fontsize=12)
axes[0].set_xlabel("Section")
axes[0].set_ylabel("Average Marks")

# 2 - bubble: area encodes the number (hard to compare)
sizes = [m * 14 for m in marks]
sc = axes[1].scatter(range(len(sections)), [1] * len(sections), s=sizes,
                     c=marks, cmap="Oranges", alpha=0.75, edgecolors="black")
for i, m in enumerate(marks):
    axes[1].annotate(f"{m:.1f}", (i, 1), ha="center", va="center",
                     fontweight="bold", fontsize=9)
axes[1].set_xticks(range(len(sections)))
axes[1].set_xticklabels(sections)
axes[1].set_yticks([])
axes[1].set_title("2. Area + Colour\n(weak for exact values)", fontsize=12)
axes[1].set_xlabel("Section")
fig.colorbar(sc, ax=axes[1], label="Marks")

# 3 - heatmap: colour intensity only (worst for precise reading)
heat = np.array(marks).reshape(1, -1)
im = axes[2].imshow(heat, cmap="YlOrRd", aspect="auto", vmin=0, vmax=100)
axes[2].set_xticks(range(len(sections)))
axes[2].set_xticklabels(sections)
axes[2].set_yticks([])
axes[2].set_title("3. Colour intensity\n(least accurate)", fontsize=12)
axes[2].set_xlabel("Section")
for i, m in enumerate(marks):
    axes[2].text(i, 0, f"{m:.1f}", ha="center", va="center", fontweight="bold")
fig.colorbar(im, ax=axes[2], label="Marks")

fig.suptitle("Same Data, Three Encodings", fontsize=15)
fig.tight_layout()
fig.savefig("q04_position_vs_size.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# CLEVELAND & McGILL HIERARCHY (1984 experiment on graphical perception)
# ---------------------------------------------------------------------------
ranking = [
    (1, "Position on common scale", "Bar chart / dot plot - error ~5%"),
    (2, "Position on non-aligned scale", "Two separate axes - error ~10%"),
    (3, "Length", "Bars, line segments - error ~13%"),
    (4, "Direction / angle", "Slopes - error ~20%"),
    (5, "Area (size)", "Bubbles, circles - error ~27%"),
    (6, "Volume / 3D", "3D shapes - error >35%"),
    (7, "Colour saturation / hue", "Heatmaps - worst for quantity"),
]
print("\nCLEVELAND & McGILL EFFECTIVENESS RANKING")
for rank, name, note in ranking:
    print(f"  {rank}. {name:30s} <- {note}")

print("\nWHY POSITION BEATS SIZE")
print("- Human area judgement is compressive (Stevens' power law: perceived")
print("  area ~ physical area^0.7), so doubling a number does NOT look doubled.")
print("- A 58 vs 62 bubble pair is almost indistinguishable, while two bars on")
print("  a common axis show the same gap instantly and accurately.")

# ---------------------------------------------------------------------------
# REDESIGN of the least effective encoding (heatmap -> bar + value labels)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# old: colour-only heatmap
sns.heatmap(heat, annot=True, fmt=".1f", cmap="YlOrRd", vmin=0, vmax=100,
            xticklabels=sections, yticklabels=["avg marks"], ax=axes[0],
            cbar_kws={"label": "Marks"})
axes[0].set_title("Before: heatmap (colour only)", fontsize=12)

# new: position-based bars with labels
axes[1].bar(sections, marks, color="#2196F3", edgecolor="black", width=0.6)
for i, v in enumerate(marks):
    axes[1].text(i, v + 1, f"{v:.1f}", ha="center", fontweight="bold")
axes[1].set_ylim(0, 100)
axes[1].axhline(np.mean(marks), color="#FF5722", linestyle="--",
                label=f"Overall mean {np.mean(marks):.1f}")
axes[1].set_title("After: position on a common scale", fontsize=12)
axes[1].set_xlabel("Section")
axes[1].set_ylabel("Average Marks")
axes[1].legend()

fig.suptitle("Redesign: from colour intensity to position", fontsize=14)
fig.tight_layout()
fig.savefig("q04_redesign.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
print("\nINTERPRETATION")
print(f"- Section averages: " + ", ".join(f"{s}={m:.1f}" for s, m in zip(sections, marks)))
print("- All three charts show the SAME numbers, but only the bar chart lets you")
print("  read differences accurately without a colour legend.")
print("- Rule of thumb: encode quantities with position/length; use colour for")
print("  categories or emphasis, not for precise values.")

## Question 5: Exploratory Data Analysis

**Concepts:** distributions, skewness, IQR outliers, KDE/violin plots.

In [ ]:
# --- Q5: four views of the key variables ---
cols = ["attendance_percentage", "study_hours_per_week", "lab_marks",
        "internal_marks", "final_exam_marks"]
short = ["Attendance", "Study Hours", "Lab Marks", "Internal", "Final Marks"]

# histograms
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, c in enumerate(cols):
    ax = axes.ravel()[i]
    ax.hist(df[c], bins=20, color="#42A5F5", edgecolor="black", alpha=0.85)
    ax.set_title(short[i])
axes.ravel()[5].set_visible(False)
fig.suptitle("Histograms of Key Variables")
plt.tight_layout(); plt.savefig("q05_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

# box plots
plt.figure(figsize=(10, 5)); df[cols].boxplot(rot=15)
plt.title("Box Plots of Key Variables"); plt.ylabel("Value")
plt.tight_layout(); plt.savefig("q05_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

# density plots
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, c in enumerate(cols):
    sns.kdeplot(df[c], ax=axes[i], fill=True, color="#2196F3")
    axes[i].set_title(short[i])
plt.tight_layout(); plt.savefig("q05_density.png", dpi=150, bbox_inches="tight")
plt.show()

# violin plots
long = df[cols].melt(var_name="variable", value_name="value")
plt.figure(figsize=(12, 5))
sns.violinplot(x="variable", y="value", data=long, palette="Set2", inner="quartile")
plt.title("Violin Plots of Key Variables"); plt.xticks(rotation=15)
plt.tight_layout(); plt.savefig("q05_violin.png", dpi=150, bbox_inches="tight")
plt.show()

# statistics table
for c in cols:
    q1, q3 = df[c].quantile(.25), df[c].quantile(.75)
    iqr = q3 - q1
    n_out = ((df[c] < q1 - 1.5*iqr) | (df[c] > q3 + 1.5*iqr)).sum()
    print(f"{c:26s} mean={df[c].mean():6.2f} median={df[c].median():6.2f} "
          f"std={df[c].std():5.2f} skew={df[c].skew():+5.2f} IQR-outliers={n_out}")

**Interpretation:** `study_hours` is right-skewed (few hard workers), `lab_marks` is
left-skewed (most do fine), and `final_exam_marks` is nearly symmetric — the classic bell shape.

### 🔬 Advanced version — `full_version/q05_advanced_eda.py`

Full exploratory data analysis: histograms with KDE, box plots with mean
markers, violin plots with embedded boxes, QQ plots for normality, a
formatted statistics table (mean/median/mode/std/skew/kurtosis) and
outlier detection by BOTH the IQR and z-score methods.


In [ ]:
"""
Question 5: EDA (Comprehensive Statistical)
Advanced Version — Data Visualization Lab
Author: Arul Agnes
Date: 2025

Full exploratory data analysis: histograms with KDE, box plots with mean
markers, violin plots with embedded boxes, QQ plots for normality, a
formatted statistics table (mean/median/mode/std/skew/kurtosis) and
outlier detection by BOTH the IQR and z-score methods.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")

cols = ["attendance_percentage", "study_hours_per_week", "lab_marks",
        "internal_marks", "final_exam_marks"]
short = ["Attendance", "Study Hours", "Lab Marks", "Internal", "Final Marks"]

# ---------------------------------------------------------------------------
# FIGURE 1 - COMBINED EDA (2x2: hist+KDE, box with means, violin, QQ)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) histogram + KDE overlay for every variable (melted frame)
long = df[cols].melt(var_name="variable", value_name="value")
sns.histplot(data=long, x="value", hue="variable", element="step",
             stat="density", common_norm=False, kde=True, ax=axes[0, 0],
             palette="Set2")
axes[0, 0].set_title("(a) Histograms with KDE overlay", fontsize=12)
axes[0, 0].set_xlabel("Value")
axes[0, 0].legend(fontsize=7, title="")

# (b) box plots with explicit mean markers
sns.boxplot(data=df[cols], orient="h", palette="Set2", ax=axes[0, 1])
means = df[cols].mean()
for i, m in enumerate(means):
    axes[0, 1].plot(m, i, marker="D", color="red", markersize=7, zorder=5)
axes[0, 1].set_title("(b) Box plots (red diamond = mean)", fontsize=12)
axes[0, 1].set_xlabel("Value")

# (c) violin plots with embedded mini box plots
sns.violinplot(data=long, x="variable", y="value", inner="box",
               palette="Set2", ax=axes[1, 0], cut=0)
axes[1, 0].set_title("(c) Violin plots with inner box", fontsize=12)
axes[1, 0].set_xlabel("")
axes[1, 0].tick_params(axis="x", rotation=15)

# (d) QQ plots: points on the line = normally distributed
for col, color in zip(cols, sns.color_palette("Set2")):
    stats.probplot(df[col].dropna(), dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("(d) Normal QQ plots (all variables)", fontsize=12)
axes[1, 1].get_lines()[0].set(marker="o", markersize=2.5, color="#2196F3")
for line in axes[1, 1].get_lines()[1:]:
    line.set_color("#FF5722")

fig.suptitle("Comprehensive EDA of Key Variables", fontsize=15)
fig.tight_layout()
fig.savefig("q05_combined_eda.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# STATISTICS TABLE
# ---------------------------------------------------------------------------
print("=" * 96)
print(f"{'Variable':24s} {'Mean':>7} {'Median':>7} {'Mode':>7} {'Std':>7} "
      f"{'Skew':>7} {'Kurt':>7} {'Normal?':>9}")
print("-" * 96)
stats_rows = []
for col in cols:
    s = df[col]
    # Shapiro test on a sample (test breaks on n>5000; 1200 is fine)
    _, p = stats.shapiro(s.sample(min(len(s), 500), random_state=42))
    normal = "yes" if p > 0.05 else "no"
    row = dict(var=col, mean=s.mean(), median=s.median(),
               mode=s.mode().iloc[0], std=s.std(),
               skew=s.skew(), kurt=s.kurtosis(), p=p, normal=normal)
    stats_rows.append(row)
    print(f"{col:24s} {row['mean']:7.2f} {row['median']:7.2f} {row['mode']:7.2f} "
          f"{row['std']:7.2f} {row['skew']:7.2f} {row['kurt']:7.2f} {normal:>9}")
print("-" * 96)
print("(Shapiro-Wilk normality test on 500-row sample; p>0.05 => compatible with normal)")

# ---------------------------------------------------------------------------
# OUTLIER DETECTION: IQR rule vs z-score rule
# ---------------------------------------------------------------------------
iqr_counts, z_counts = {}, {}
for col in cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    iqr_counts[col] = int(((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum())
    z = np.abs((df[col] - df[col].mean()) / df[col].std())
    z_counts[col] = int((z > 3).sum())

out_tbl = pd.DataFrame({"IQR_method": iqr_counts, "zscore_method": z_counts})
out_tbl["difference"] = out_tbl["IQR_method"] - out_tbl["zscore_method"]
print("\nOUTLIER DETECTION COMPARISON")
print(out_tbl.to_string())

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(cols))
w = 0.36
ax.bar(x - w / 2, out_tbl["IQR_method"], w, label="IQR (Q1-1.5*IQR fences)",
       color="#2196F3", edgecolor="black")
ax.bar(x + w / 2, out_tbl["zscore_method"], w, label="z-score (|z| > 3)",
       color="#FF5722", edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels(short)
ax.set_ylabel("Number of Outliers")
ax.set_title("Outlier Counts: IQR vs z-score", fontsize=13)
ax.legend()
for i, (a, b) in enumerate(zip(out_tbl["IQR_method"], out_tbl["zscore_method"])):
    ax.text(i - w / 2, a + 0.4, str(a), ha="center", fontsize=9)
    ax.text(i + w / 2, b + 0.4, str(b), ha="center", fontsize=9)
fig.tight_layout()
fig.savefig("q05_outlier_detection.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
print("\nINTERPRETATION")
for row in stats_rows:
    shape = "right-skewed" if row["skew"] > 0.5 else (
        "left-skewed" if row["skew"] < -0.5 else "roughly symmetric")
    print(f"- {row['var']}: {shape} (skew={row['skew']:.2f}), "
          f"normal? {row['normal']}")
print("- The IQR rule always flags MORE outliers than |z|>3 because it uses")
print("  quartiles instead of mean/std and reacts to skew.")
print("- QQ points hugging the red line => that variable is close to normal;")
print("  systematic S-curves or fanning => non-normal.")

## Question 6: Scatter Plot, Pair Plot and Correlation Heatmap

**Concepts:** relationships, correlation matrix, correlation ≠ causation.

In [ ]:
# --- Q6: relationships between variables ---
plt.figure(figsize=(8, 5))
sns.scatterplot(x="study_hours_per_week", y="final_exam_marks", hue="pass_status",
                data=df, palette={"Pass": "#4CAF50", "Fail": "#FF5722"}, alpha=0.7)
plt.title("Study Hours vs Final Marks")
plt.xlabel("Study Hours per Week"); plt.ylabel("Final Exam Marks")
plt.tight_layout(); plt.savefig("q06_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

g = sns.pairplot(df, hue="pass_status",
                 palette={"Pass": "#4CAF50", "Fail": "#FF5722"},
                 vars=["study_hours_per_week", "attendance_percentage",
                       "lab_marks", "final_exam_marks"], corner=True)
g.savefig("q06_pairplot.png", dpi=150, bbox_inches="tight")
plt.show()

num = df.select_dtypes(include="number")
corr = num.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            linewidths=0.5)
plt.title("Correlation Heatmap")
plt.tight_layout(); plt.savefig("q06_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

pairs = corr["final_exam_marks"].drop("final_exam_marks").sort_values(ascending=False)
print(f"Strongest positive: {pairs.index[0]} (r={pairs.iloc[0]:.2f})")
print(f"Weakest:            {pairs.index[-1]} (r={pairs.iloc[-1]:.2f})")
print("NOTE: Correlation does NOT prove causation!")

**Interpretation:** study hours show the strongest positive link with marks; support
sessions are negative because *struggling* students attend them. **Correlation ≠ causation.**

### 🔬 Advanced version — `full_version/q06_advanced_correlation.py`

Top correlated pairs with fitted regression lines and annotated
outliers, a pair plot with KDE diagonals, and an annotated heatmap with
Pearson p-values and strength labels. Includes a strong reminder that
correlation is not causation.

> Note: the advanced pair plot is saved as `q06_pairplot_adv.png` so it does not overwrite the simple version's figure.


In [ ]:
"""
Question 6: Correlation Analysis (Deep Dive)
Advanced Version — Data Visualization Lab
Author: Arul Agnes
Date: 2025

Top correlated pairs with fitted regression lines and annotated
outliers, a pair plot with KDE diagonals, and an annotated heatmap with
Pearson p-values and strength labels. Includes a strong reminder that
correlation is not causation.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")

num_cols = df.select_dtypes(include="number").columns.tolist()
corr = df[num_cols].corr()

# ---------------------------------------------------------------------------
# TOP-3 CORRELATED PAIRS WITH THE TARGET (final_exam_marks)
# ---------------------------------------------------------------------------
target = "final_exam_marks"
others = corr[target].drop(target).abs().sort_values(ascending=False)
top3 = list(others.index[:3])
print("TOP-3 PREDICTORS OF final_exam_marks:")
print(corr[target].drop(target).reindex(top3).round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, top3):
    sns.regplot(x=col, y=target, data=df, ax=ax, scatter_kws={"alpha": 0.4},
                line_kws={"color": "#FF5722"}, ci=95)
    r, p = stats.pearsonr(df[col], df[target])
    # highlight statistical outliers (|z|>3 on both axes) in red
    zx = np.abs((df[col] - df[col].mean()) / df[col].std())
    zy = np.abs((df[target] - df[target].mean()) / df[target].std())
    odd = df[(zx > 3) | (zy > 3)]
    ax.scatter(odd[col], odd[target], s=60, facecolors="none",
               edgecolors="black", linewidths=1.5, label="outliers |z|>3")
    ax.set_title(f"{col} vs {target}\nr={r:.3f}, p={'<0.001' if p < 0.001 else f'{p:.3f}'}",
                 fontsize=11)
    ax.legend(fontsize=8)
fig.suptitle("Top 3 Correlated Pairs with Final Exam Marks", fontsize=14)
fig.tight_layout()
fig.savefig("q06_advanced_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# PAIR PLOT with KDE on the diagonal, coloured by department
# ---------------------------------------------------------------------------
pair_vars = ["study_hours_per_week", "attendance_percentage",
             "lab_marks", "final_exam_marks"]
g = sns.pairplot(df, vars=pair_vars, hue="department", palette="Set1",
                 diag_kind="kde", plot_kws={"alpha": 0.5, "s": 22})
g.figure.suptitle("Pair Plot (KDE diagonal) by Department", y=1.02)
g.savefig("q06_pairplot_adv.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# ANNOTATED HEATMAP: r values + significance stars
# ---------------------------------------------------------------------------
pvals = pd.DataFrame(np.ones((len(num_cols), len(num_cols))),
                     index=num_cols, columns=num_cols)
for a in num_cols:
    for b in num_cols:
        pvals.loc[a, b] = stats.pearsonr(df[a], df[b])[1]


def strength(r):
    """Label a correlation coefficient by strength."""
    ar = abs(r)
    if ar > 0.6:
        return "S"
    if ar > 0.3:
        return "M"
    return "W"


labels = corr.round(2).astype(str)
for a in num_cols:
    for b in num_cols:
        star = "***" if pvals.loc[a, b] < 0.001 else (
            "**" if pvals.loc[a, b] < 0.01 else (
                "*" if pvals.loc[a, b] < 0.05 else "ns"))
        labels.loc[a, b] = f"{corr.loc[a, b]:.2f}\n{star}"

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=labels, fmt="", cmap="coolwarm", vmin=-1, vmax=1,
            linewidths=0.5, square=True,
            annot_kws={"size": 7})
plt.title("Correlation Heatmap with Pearson p-values\n"
          "(*** p<0.001, ** p<0.01, * p<0.05, ns = not significant)", fontsize=12)
plt.tight_layout()
plt.savefig("q06_annotated_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# COMPREHENSIVE REPORT
# ---------------------------------------------------------------------------
print("\nCOMPREHENSIVE CORRELATION REPORT")
print(f"{'Pair':56s} {'r':>7} {'p':>9} {'strength':>9}")
pairs_done = set()
for a in num_cols:
    for b in num_cols:
        if a >= b or (a, b) in pairs_done:
            continue
        pairs_done.add((a, b))
        r, p = corr.loc[a, b], pvals.loc[a, b]
        if abs(r) > 0.3:
            sig = "<0.001" if p < 0.001 else f"{p:.4f}"
            print(f"{a + ' <-> ' + b:56s} {r:+7.3f} {sig:>9} {strength(r):>9}")

print("\nSIGN LEGEND: S = strong (|r|>0.6), M = moderate (0.3-0.6), W = weak (<0.3)")
print("\n*** CORRELATION IS NOT CAUSATION ***")
print("- High study hours and high marks move together, but this data is")
print("  observational: a hidden factor (student ability) could drive both.")
print("- To claim causation you would need a controlled experiment or a")
print("  carefully justified causal model, not just a large r value.")
print("- Also beware: support sessions correlate NEGATIVELY with marks, yet")
print("  sessions HELP - struggling students are the ones attending them.")

## Question 7: Linear Regression

**Concepts:** fit line, slope/intercept, R², residuals.

In [ ]:
# --- Q7: regress marks on study hours ---
X = df[["study_hours_per_week"]]
y = df["final_exam_marks"]
model = LinearRegression().fit(X, y)
slope, intercept, r2 = model.coef_[0], model.intercept_, model.score(X, y)
y_pred = model.predict(X)

plt.figure(figsize=(8, 5))
sns.regplot(x="study_hours_per_week", y="final_exam_marks", data=df, ci=95,
            scatter_kws={"alpha": 0.4, "color": "#2196F3"},
            line_kws={"color": "#FF5722"})
plt.title("Linear Regression: Study Hours -> Final Marks")
plt.xlabel("Study Hours per Week"); plt.ylabel("Final Exam Marks")
plt.text(0.05, 0.95, f"Slope = {slope:.2f}\nIntercept = {intercept:.1f}\nR2 = {r2:.3f}",
         transform=plt.gca().transAxes, va="top",
         bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
plt.tight_layout(); plt.savefig("q07_regression.png", dpi=150, bbox_inches="tight")
plt.show()

residuals = y - y_pred
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.5, color="#4CAF50")
plt.axhline(0, color="red", linestyle="--", linewidth=2)
plt.title("Residual Plot"); plt.xlabel("Predicted Marks")
plt.ylabel("Residual (Actual - Predicted)")
plt.tight_layout(); plt.savefig("q07_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Slope: {slope:.2f} marks per extra study hour")
print(f"Intercept: {intercept:.1f} (predicted marks at 0 hours)")
print(f"R-squared: {r2:.3f} -> study hours explain {r2*100:.1f}% of the variance")
print("Limitation: correlation is not causation; residuals scattered = good fit.")

**Interpretation:** every extra study hour is worth ~0.9 marks on average, but R² ≈ 0.32
means ~68% of the variation comes from other factors (attendance, prior ability, luck…).

### 🔬 Advanced version — `full_version/q07_advanced_regression.py`

Fits the study-hours -> marks model with BOTH scikit-learn and
statsmodels, prints full OLS statistics (SE, t, p, R2, adjusted R2),
draws a 2x2 residual diagnostics panel, Cook's distance, and separates
the confidence interval from the prediction interval.


In [ ]:
"""
Question 7: Regression Diagnostics (Full Analysis)
Advanced Version — Data Visualization Lab
Author: Arul Agnes
Date: 2025

Fits the study-hours -> marks model with BOTH scikit-learn and
statsmodels, prints full OLS statistics (SE, t, p, R2, adjusted R2),
draws a 2x2 residual diagnostics panel, Cook's distance, and separates
the confidence interval from the prediction interval.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from scipy import stats
import statsmodels.api as sm

sns.set_style("whitegrid")

X = df[["study_hours_per_week"]]
y = df["final_exam_marks"]

# ---------------------------------------------------------------------------
# 1. scikit-learn fit (simple API)
# ---------------------------------------------------------------------------
sk_model = LinearRegression().fit(X, y)
sk_slope, sk_intercept = sk_model.coef_[0], sk_model.intercept_
sk_r2 = sk_model.score(X, y)
y_pred = sk_model.predict(X)
residuals = y - y_pred

# ---------------------------------------------------------------------------
# 2. statsmodels OLS fit (full statistics)
# ---------------------------------------------------------------------------
Xc = sm.add_constant(X)                      # adds the intercept column
ols = sm.OLS(y, Xc).fit()
print("=" * 70)
print("STATSMODELS OLS SUMMARY (key numbers)")
print("=" * 70)
print(ols.summary2().tables[1].round(4).to_string())   # coefficients table
print(f"R-squared      : {ols.rsquared:.4f}")
print(f"Adj. R-squared : {ols.rsquared_adj:.4f}")
print(f"F-statistic    : {ols.fvalue:.1f} (p = {ols.f_pvalue:.2e})")
print(f"\nMODEL EQUATION: final_marks = {ols.params.iloc[1]:.3f} * study_hours "
      f"+ {ols.params.iloc[0]:.2f}")

# ---------------------------------------------------------------------------
# 3. 2x2 RESIDUAL DIAGNOSTICS
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# (a) residuals vs fitted - should be random scatter around 0
axes[0, 0].scatter(y_pred, residuals, alpha=0.4, color="#2196F3", s=20)
axes[0, 0].axhline(0, color="red", linewidth=2)
axes[0, 0].set(title="(a) Residuals vs Fitted", xlabel="Fitted values",
               ylabel="Residuals")
# lowess smoother reveals systematic patterns
lowess = sm.nonparametric.lowess(residuals, y_pred, frac=0.4)
axes[0, 0].plot(lowess[:, 0], lowess[:, 1], color="#FF5722", linewidth=2)

# (b) normal QQ plot of residuals
sm.qqplot(residuals, line="45", ax=axes[0, 1], alpha=0.4)
axes[0, 1].set(title="(b) Normal QQ of Residuals")

# (c) scale-location: sqrt |standardized residuals| vs fitted
std_res = ols.get_influence().resid_studentized_internal
axes[1, 0].scatter(y_pred, np.sqrt(np.abs(std_res)), alpha=0.4,
                   color="#4CAF50", s=20)
axes[1, 0].set(title="(c) Scale-Location (homoscedasticity)",
               xlabel="Fitted values", ylabel="sqrt(|std residuals|)")

# (d) residual histogram + fitted normal curve
axes[1, 1].hist(residuals, bins=25, density=True, color="#FF9800",
                edgecolor="black", alpha=0.85)
xs = np.linspace(residuals.min(), residuals.max(), 200)
axes[1, 1].plot(xs, stats.norm.pdf(xs, residuals.mean(), residuals.std()),
                color="black", linewidth=2, label="normal fit")
axes[1, 1].set(title="(d) Residual Distribution", xlabel="Residual")
axes[1, 1].legend()

fig.suptitle("Regression Diagnostics: study_hours -> final_marks", fontsize=15)
fig.tight_layout()
fig.savefig("q07_regression_diagnostics.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# 4. CONFIDENCE INTERVAL vs PREDICTION INTERVAL
# ---------------------------------------------------------------------------
grid = np.linspace(X["study_hours_per_week"].min(),
                   X["study_hours_per_week"].max(), 100).reshape(-1, 1)
grid_pred = sm.add_constant(grid)
mean_ci = ols.get_prediction(grid_pred).conf_int(alpha=0.05)      # E[Y|X]
indiv_pi = ols.get_prediction(grid_pred).conf_int(alpha=0.05, obs=True)  # new Y

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X["study_hours_per_week"], y, alpha=0.35, s=18, color="#2196F3",
           label="students")
ax.plot(grid, ols.params.iloc[0] + ols.params.iloc[1] * grid,
        color="#FF5722", linewidth=2, label="fitted line")
ax.fill_between(grid.ravel(), mean_ci[:, 0], mean_ci[:, 1], alpha=0.45,
                color="#FF5722", label="95% confidence interval (mean)")
ax.fill_between(grid.ravel(), indiv_pi[:, 0], indiv_pi[:, 1], alpha=0.18,
                color="#9C27B0", label="95% prediction interval (individual)")
ax.set_xlabel("Study Hours per Week")
ax.set_ylabel("Final Exam Marks")
ax.set_title("Confidence Interval vs Prediction Interval", fontsize=13)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig("q07_confidence_prediction.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# 5. COOK'S DISTANCE - influential observations
# ---------------------------------------------------------------------------
cooks = ols.get_influence().cooks_distance[0]
n_influential = int((cooks > 4 / len(y)).sum())
influential_idx = np.where(cooks > 4 / len(y))[0]

# ---------------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------------
print("\nDIAGNOSTICS REPORT")
print(f"- sklearn slope {sk_slope:.3f} == statsmodels slope "
      f"{ols.params.iloc[1]:.3f} (both agree)")
print(f"- Residual std: {residuals.std():.2f} marks")
shap_stat, shap_p = stats.shapiro(residuals.sample(500, random_state=42))
print(f"- Residual normality (Shapiro, n=500): p = {shap_p:.3f} "
      f"-> {'normal-ish' if shap_p > 0.05 else 'non-normal'}")
print(f"- Cook's distance: {n_influential} points above 4/n "
      f"({4 / len(y):.4f}) are influential")
print("\nASSUMPTIONS CHECK")
print("1. LINEARITY     : lowess line in plot (a) is roughly flat => OK")
print("2. INDEPENDENCE  : data is one row per student => plausible")
print("3. NORMALITY     : QQ points near the 45-degree line => OK")
print("4. EQUAL VARIANCE : plot (c) shows no funnel => OK")
print("\nINTERPRETATION")
print(f"- The model explains {sk_r2 * 100:.1f}% of the variance; the other "
      f"{100 - sk_r2 * 100:.1f}% comes from attendance, prior ability, luck...")
print("- Confidence interval = uncertainty of the MEAN response (narrow band)")
print("- Prediction interval = uncertainty for ONE student (much wider band)")
print("- A narrow CI does NOT mean individual predictions are accurate!")

## Question 8: Rolling Average and Uncertainty Band

**Concepts:** smoothing, standard deviation band, trend vs noise.

In [ ]:
# --- Q8: rolling average + uncertainty band ---
weekly = (df.groupby("assessment_date")["final_exam_marks"]
            .agg(mean="mean", std="std").reset_index().sort_values("assessment_date"))
weekly["roll"] = weekly["mean"].rolling(3).mean()
weekly["roll_std"] = weekly["std"].rolling(3).std()

plt.figure(figsize=(12, 5))
plt.plot(weekly["assessment_date"], weekly["mean"], marker="o", linewidth=1.2,
         alpha=0.6, label="Weekly Mean")
plt.plot(weekly["assessment_date"], weekly["roll"], linewidth=2.5,
         color="#FF5722", label="3-Week Rolling Average")
plt.fill_between(weekly["assessment_date"],
                 weekly["roll"] - weekly["roll_std"],
                 weekly["roll"] + weekly["roll_std"],
                 alpha=0.3, color="#FF5722", label="+/-1 Std Dev Band")
plt.title("Performance Trend with Rolling Average and Uncertainty Band")
plt.xlabel("Assessment Date"); plt.ylabel("Average Final Exam Marks")
plt.xticks(rotation=45); plt.legend()
plt.tight_layout(); plt.savefig("q08_rolling_avg.png", dpi=150, bbox_inches="tight")
plt.show()

w = weekly.loc[weekly["std"].idxmax()]
print("Rolling average smooths weekly fluctuations to reveal the trend.")
print(f"Widest spread: {w['assessment_date'].date()} (std = {w['std']:.1f})")
print("The band is +/-1 standard deviation (data spread), NOT a confidence interval.")

**Interpretation:** wide bands mean the cohort tested that day was mixed; the rolling
line is the trend you should quote when someone asks "are marks improving?" 

### 🔬 Advanced version — `full_version/q08_advanced_rolling_uncertainty.py`

Weekly means with standard errors and 95% confidence intervals, a
4-week rolling average with BOTH a +/-1 std band and a 95% CI band,
error bars, and a comparison of 2/4/6-week smoothing windows.


In [ ]:
"""
Question 8: Time Series Uncertainty (Detailed)
Advanced Version — Data Visualization Lab
Author: Arul Agnes
Date: 2025

Weekly means with standard errors and 95% confidence intervals, a
4-week rolling average with BOTH a +/-1 std band and a 95% CI band,
error bars, and a comparison of 2/4/6-week smoothing windows.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

df["assessment_date"] = pd.to_datetime(df["assessment_date"])
sns.set_style("whitegrid")

# ---------------------------------------------------------------------------
# Weekly aggregation: mean, std, count, standard error, 95% CI
# ---------------------------------------------------------------------------
wk = (df.groupby("assessment_date")["final_exam_marks"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
        .sort_values("assessment_date"))
wk["se"] = wk["std"] / np.sqrt(wk["n"])                 # standard error of mean
wk["ci_lo"] = wk["mean"] - 1.96 * wk["se"]              # 95% confidence interval
wk["ci_hi"] = wk["mean"] + 1.96 * wk["se"]
wk["sd_lo"] = wk["mean"] - wk["std"]                    # +/-1 std dev band
wk["sd_hi"] = wk["mean"] + wk["std"]
wk["roll4"] = wk["mean"].rolling(4).mean()              # 4-week rolling mean

print("WEEKLY STATISTICS")
print(wk[["assessment_date", "mean", "std", "n", "se", "ci_lo", "ci_hi"]]
      .round(2).to_string(index=False))

# ---------------------------------------------------------------------------
# FIGURE 1 - UNCERTAINTY BANDS (std band vs 95% CI band)
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(wk["assessment_date"], wk["mean"], marker="o", linewidth=1.3,
        alpha=0.65, color="#2196F3", label="Weekly mean")
ax.plot(wk["assessment_date"], wk["roll4"], linewidth=2.6, color="#FF5722",
        label="4-week rolling average")
ax.fill_between(wk["assessment_date"], wk["sd_lo"], wk["sd_hi"],
                alpha=0.18, color="#4CAF50", label="+/-1 std dev (data spread)")
ax.fill_between(wk["assessment_date"], wk["ci_lo"], wk["ci_hi"],
                alpha=0.4, color="#FF5722", label="95% CI of the mean")
ax.set_title("Weekly Performance: Rolling Average, Std Band and 95% Confidence Band",
             fontsize=13)
ax.set_xlabel("Assessment Date")
ax.set_ylabel("Final Exam Marks")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax.legend(loc="lower left", fontsize=9)
fig.tight_layout()
fig.savefig("q08_uncertainty_band.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# FIGURE 2 - ERROR BARS + WINDOW COMPARISON
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# (a) error bars showing the 95% CI at every point
axes[0].errorbar(wk["assessment_date"], wk["mean"], yerr=1.96 * wk["se"],
                 fmt="o", capsize=4, markersize=5, color="#2196F3",
                 ecolor="#FF5722", elinewidth=1.5,
                 label="mean +/- 1.96 SE")
axes[0].set_title("(a) Weekly means with 95% CI error bars", fontsize=12)
axes[0].set_xlabel("Assessment Date")
axes[0].set_ylabel("Mean Marks")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha="right")
axes[0].legend(fontsize=9)

# (b) same series smoothed with 2-, 4- and 6-week windows
for window, color in [(2, "#9C27B0"), (4, "#FF5722"), (6, "#2196F3")]:
    axes[1].plot(wk["assessment_date"], wk["mean"].rolling(window).mean(),
                 linewidth=2, color=color, label=f"{window}-week rolling")
axes[1].plot(wk["assessment_date"], wk["mean"], linewidth=1, alpha=0.4,
             color="gray", label="raw weekly mean")
axes[1].set_title("(b) Rolling window comparison", fontsize=12)
axes[1].set_xlabel("Assessment Date")
axes[1].set_ylabel("Mean Marks")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha="right")
axes[1].legend(fontsize=9)

fig.tight_layout()
fig.savefig("q08_error_bars.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------------
widest = wk.loc[wk["std"].idxmax()]
narrowest = wk.loc[wk["se"].idxmax()]
print("\nINTERPRETATION")
print("- Std dev shows how SPREAD OUT the students were on that date.")
print("- The 95% CI shows how precisely the MEAN is estimated: "
      "mean +/- 1.96 * std/sqrt(n).")
print(f"- Widest spread: {widest['assessment_date'].date()} "
      f"(std = {widest['std']:.1f}, n = {int(widest['n'])})")
print(f"- Widest CI (least precise mean): {narrowest['assessment_date'].date()} "
      f"(+/- {1.96 * narrowest['se']:.1f} marks)")
print("- The std band is much wider than the CI band: individual students vary a")
print("  lot, but a mean of ~60 students is estimated quite precisely.")
print("- 2-week windows track noise; 6-week windows hide real turning points -")
print("  pick the window based on how smooth your question needs the answer to be.")

## Question 9: PCA

**Concepts:** standardization, explained variance, dimensionality reduction.

In [ ]:
# --- Q9: PCA from 11 features to 2 dimensions ---
features = ["attendance_percentage", "study_hours_per_week", "lab_marks",
            "assignment_marks", "internal_marks", "previous_semester_cgpa",
            "online_assessment_score", "library_visits_per_month",
            "coding_practice_hours_per_week", "support_sessions_attended",
            "final_exam_marks"]
X = df[features]
print("Why standardize? PCA is scale-sensitive: 0-100 would dominate 0-10.")
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
emb = pca.fit_transform(X_scaled)
print(f"PC1 explained variance: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"PC2 explained variance: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"Cumulative:            {pca.explained_variance_ratio_.sum()*100:.1f}%")

order = ["O", "A+", "A", "B+", "B", "C", "F"]
plt.figure(figsize=(9, 6))
sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=df["final_grade"], hue_order=order,
                palette="viridis", alpha=0.75, s=45)
plt.title("PCA: Students in 2D Space (colored by Grade)")
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.legend(title="Grade", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.savefig("q09_pca_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(["PC1", "PC2"], pca.explained_variance_ratio_ * 100,
        color=["#2196F3", "#4CAF50"], edgecolor="black")
plt.title("Explained Variance per Component"); plt.ylabel("Variance Explained (%)")
plt.ylim(0, 100)
plt.tight_layout(); plt.savefig("q09_variance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Students with grade O cluster on the high side of PC1.")

**Interpretation:** PC1 is an "overall performance" axis — high marks, internals and
study hours push one way. Two PCs keep most but **not all** of the information.

### 🔬 Advanced version — `full_version/q09_advanced_pca.py`

Full PCA pipeline: all components, scree plot with Kaiser criterion,
cumulative variance with 80%/95% thresholds, a biplot with loading
vectors, and a 3D scatter of the first three components.


In [ ]:
"""
Question 9: PCA (Comprehensive)
Advanced Version — Data Visualization Lab
Author: Arul Agnes
Date: 2025

Full PCA pipeline: all components, scree plot with Kaiser criterion,
cumulative variance with 80%/95% thresholds, a biplot with loading
vectors, and a 3D scatter of the first three components.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D   # registers the 3D projection

sns.set_style("whitegrid")

features = ["attendance_percentage", "study_hours_per_week", "lab_marks",
            "assignment_marks", "internal_marks", "previous_semester_cgpa",
            "online_assessment_score", "library_visits_per_month",
            "coding_practice_hours_per_week", "support_sessions_attended",
            "final_exam_marks"]
X = df[features]
X_scaled = StandardScaler().fit_transform(X)      # PCA needs standardized data

# ---------------------------------------------------------------------------
# PCA with ALL 11 components (not just 2)
# ---------------------------------------------------------------------------
pca = PCA(n_components=len(features))
scores = pca.fit_transform(X_scaled)
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)
n80 = int(np.searchsorted(cum, 0.80) + 1)         # components for 80% variance
n95 = int(np.searchsorted(cum, 0.95) + 1)         # components for 95% variance
kaiser = int((pca.explained_variance_ > 1).sum())  # Kaiser: eigenvalue > 1

print("=" * 60)
print("PCA - EXPLAINED VARIANCE")
print("=" * 60)
for i, (v, c) in enumerate(zip(evr, cum), 1):
    print(f"  PC{i:<2d} {v * 100:5.1f}%   cumulative {c * 100:5.1f}%")
print(f"- Kaiser criterion (eigenvalue>1): keep {kaiser} components")
print(f"- {n80} components needed for 80% variance")
print(f"- {n95} components needed for 95% variance")
print(f"- 11 features -> 2 PCs retains {cum[1] * 100:.1f}% "
      f"(loses {(1 - cum[1]) * 100:.1f}%)")

# ---------------------------------------------------------------------------
# FIGURE 1 - SCREE PLOT + CUMULATIVE VARIANCE
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
comps = np.arange(1, len(features) + 1)

axes[0].bar(comps, evr * 100, color="#2196F3", edgecolor="black",
            label="variance explained")
axes[0].plot(comps, pca.explained_variance_, "o-", color="#FF5722",
             label="eigenvalue (right of Kaiser line = 1)", markersize=5)
axes[0].axhline(1, color="gray", linestyle="--", linewidth=1.5)
axes[0].set_title("Scree Plot (Kaiser criterion: eigenvalue > 1)", fontsize=12)
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Variance Explained (%) / Eigenvalue")
axes[0].legend(fontsize=9)

axes[1].plot(comps, cum * 100, "o-", color="#4CAF50", markersize=5)
axes[1].axhline(80, color="orange", linestyle="--", label="80% threshold")
axes[1].axhline(95, color="red", linestyle="--", label="95% threshold")
axes[1].axvline(n80, color="orange", linestyle=":", alpha=0.6)
axes[1].axvline(n95, color="red", linestyle=":", alpha=0.6)
axes[1].set_title("Cumulative Explained Variance", fontsize=12)
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Variance (%)")
axes[1].legend(fontsize=9)
fig.tight_layout()
fig.savefig("q09_scree_plot.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# FIGURE 2 - BIPLOT + 3D SCATTER
# ---------------------------------------------------------------------------
fig = plt.figure(figsize=(16, 6))
ax1 = fig.add_subplot(1, 2, 1)

grade_order = ["O", "A+", "A", "B+", "B", "C", "F"]
palette = dict(zip(grade_order, sns.color_palette("viridis", len(grade_order))))
for grade in grade_order:
    mask = df["final_grade"] == grade
    ax1.scatter(scores[mask, 0], scores[mask, 1], s=22, alpha=0.65,
                color=palette[grade], label=grade, edgecolors="none")

# loading vectors: show which original features push students in each direction
loadings = pca.components_[:2].T                 # 11 features x 2 PCs
scale = 4.0                                     # arrows scaled for visibility
for i, feat in enumerate(features):
    ax1.arrow(0, 0, loadings[i, 0] * scale, loadings[i, 1] * scale,
              color="#FF5722", width=0.03, head_width=0.35, alpha=0.85)
    ax1.text(loadings[i, 0] * scale * 1.15, loadings[i, 1] * scale * 1.15,
             feat.replace("_per_week", "").replace("_percentage", ""),
             fontsize=7.5, ha="center", color="black")
ax1.set_xlabel(f"PC1 ({evr[0] * 100:.1f}% variance)")
ax1.set_ylabel(f"PC2 ({evr[1] * 100:.1f}% variance)")
ax1.set_title("Biplot: scores + feature loadings", fontsize=12)
ax1.axhline(0, color="gray", linewidth=0.5)
ax1.axvline(0, color="gray", linewidth=0.5)
ax1.legend(title="Grade", fontsize=8, loc="lower right")

# 3D scatter of the first three components
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
for grade in grade_order:
    mask = df["final_grade"] == grade
    ax2.scatter(scores[mask, 0], scores[mask, 1], scores[mask, 2],
                s=18, alpha=0.7, color=palette[grade], label=grade)
ax2.set_xlabel(f"PC1 {evr[0] * 100:.0f}%")
ax2.set_ylabel(f"PC2 {evr[1] * 100:.0f}%")
ax2.set_zlabel(f"PC3 {evr[2] * 100:.0f}%")
ax2.set_title("First 3 Principal Components", fontsize=12)

fig.tight_layout()
fig.savefig("q09_biplot.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# LOADING MATRIX + INTERPRETATION
# ---------------------------------------------------------------------------
print("\nLOADING MATRIX (first 3 components)")
load_df = pd.DataFrame(
    pca.components_[:3].T,
    index=features, columns=["PC1", "PC2", "PC3"]).round(3)
print(load_df.to_string())
print("\nTOP CONTRIBUTORS")
for pc in ["PC1", "PC2", "PC3"]:
    top = load_df[pc].abs().sort_values(ascending=False).index[:3]
    print(f"  {pc}: " + ", ".join(
        f"{f} ({load_df.loc[f, pc]:+.2f})" for f in top))

print("\nINTERPRETATION")
print(f"- PC1 ({evr[0] * 100:.1f}%) is the main 'overall performance' axis:")
print("  marks, internals and study load push one way, support sessions the other.")
print(f"- Keeping only 2 PCs retains {cum[1] * 100:.1f}% of the information -")
print(f"  {(1 - cum[1]) * 100:.1f}% is discarded for the convenience of a 2D picture.")
print("- Loadings tell you WHAT each component means; long arrows = strong influence.")

## Question 10: t-SNE and UMAP

**Concepts:** non-linear embeddings, clusters, embedding pitfalls.

In [ ]:
# --- Q10: t-SNE (and UMAP if installed) ---
try:
    from umap import UMAP
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False

X_scaled = StandardScaler().fit_transform(df[features])
order = ["O", "A+", "A", "B+", "B", "C", "F"]

emb = TSNE(n_components=2, random_state=42, perplexity=30, init="pca").fit_transform(X_scaled)
plt.figure(figsize=(9, 6))
sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=df["final_grade"], hue_order=order,
                palette="viridis", alpha=0.75, s=45)
plt.title("t-SNE: Student Performance Clusters")
plt.xlabel("t-SNE Dimension 1"); plt.ylabel("t-SNE Dimension 2")
plt.legend(title="Grade", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.savefig("q10_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

if UMAP_AVAILABLE:
    emb2 = UMAP(n_components=2, random_state=42, n_neighbors=15).fit_transform(X_scaled)
    plt.figure(figsize=(9, 6))
    sns.scatterplot(x=emb2[:, 0], y=emb2[:, 1], hue=df["final_grade"], hue_order=order,
                    palette="viridis", alpha=0.75, s=45)
    plt.title("UMAP: Student Performance Clusters")
    plt.xlabel("UMAP Dimension 1"); plt.ylabel("UMAP Dimension 2")
    plt.legend(title="Grade", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout(); plt.savefig("q10_umap.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("UMAP not installed -> run:  !pip install umap-learn")

print("t-SNE groups similar students together - but distances between clusters")
print("and cluster SIZES are NOT meaningful. Visualization only, never for clustering!")

**Interpretation:** similar grades clump together in both maps; UMAP keeps more of the
global layout, t-SNE sharper local islands. **Never measure distances on either map.**

### 🔬 Advanced version — `full_version/q10_advanced_tsne_umap.py`

Runs t-SNE at perplexity 5/30/50 and UMAP at n_neighbors 5/15/50,
builds a 2x3 comparison grid, scores each embedding with the silhouette
coefficient, and summarizes the strengths/weaknesses of both methods.


In [ ]:
"""
Question 10: t-SNE and UMAP (Comparison Study)
Advanced Version — Data Visualization Lab
Author: Arul Agnes
Date: 2025

Runs t-SNE at perplexity 5/30/50 and UMAP at n_neighbors 5/15/50,
builds a 2x3 comparison grid, scores each embedding with the silhouette
coefficient, and summarizes the strengths/weaknesses of both methods.
"""

import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")           # silence UMAP/t-SNE user warnings

try:
    from umap import UMAP
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False

sns.set_style("whitegrid")

features = ["attendance_percentage", "study_hours_per_week", "lab_marks",
            "assignment_marks", "internal_marks", "previous_semester_cgpa",
            "online_assessment_score", "library_visits_per_month",
            "coding_practice_hours_per_week", "support_sessions_attended",
            "final_exam_marks"]
X = StandardScaler().fit_transform(df[features])
labels = df["final_grade"].values            # used as reference groups for scoring
grade_order = ["O", "A+", "A", "B+", "B", "C", "F"]


def silhouette(emb):
    """Silhouette of the embedding, using final_grade as reference labels.
    Higher = grade groups are better separated in that 2D view."""
    return silhouette_score(emb, labels)


# ---------------------------------------------------------------------------
# RUN BOTH METHODS ACROSS THEIR KEY PARAMETER
# ---------------------------------------------------------------------------
TSNE_PARAMS = [5, 30, 50]          # perplexity values
UMAP_PARAMS = [5, 15, 50]          # n_neighbors values

tsne_embs, tsne_scores = [], []
print("Running t-SNE (3 fits, ~5-15 s)...")
for perp in TSNE_PARAMS:
    emb = TSNE(n_components=2, random_state=42, perplexity=perp,
               init="pca").fit_transform(X)
    tsne_embs.append(emb)
    tsne_scores.append(silhouette(emb))

umap_embs, umap_scores = [], []
if UMAP_AVAILABLE:
    print("Running UMAP (3 fits, ~10-30 s the first time)...")
    for nn in UMAP_PARAMS:
        emb = UMAP(n_components=2, random_state=42, n_neighbors=nn).fit_transform(X)
        umap_embs.append(emb)
        umap_scores.append(silhouette(emb))
else:
    print("UMAP not installed -> pip install umap-learn  (Kaggle: !pip install umap-learn)")

# ---------------------------------------------------------------------------
# FIGURE 1 - 2x3 COMPARISON GRID
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for i, (perp, emb, score) in enumerate(zip(TSNE_PARAMS, tsne_embs, tsne_scores)):
    sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=df["final_grade"],
                    hue_order=grade_order, palette="viridis",
                    s=22, alpha=0.7, legend=(i == 0), ax=axes[0, i])
    axes[0, i].set_title(f"t-SNE perplexity={perp}\nsilhouette={score:.3f}",
                         fontsize=11)
    axes[0, i].set_xlabel("dim 1")
    axes[0, i].set_ylabel("dim 2")

if UMAP_AVAILABLE:
    for i, (nn, emb, score) in enumerate(zip(UMAP_PARAMS, umap_embs, umap_scores)):
        sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=df["final_grade"],
                        hue_order=grade_order, palette="viridis",
                        s=22, alpha=0.7, legend=(i == 0), ax=axes[1, i])
        axes[1, i].set_title(f"UMAP n_neighbors={nn}\nsilhouette={score:.3f}",
                             fontsize=11)
        axes[1, i].set_xlabel("dim 1")
        axes[1, i].set_ylabel("dim 2")
else:
    for i, nn in enumerate(UMAP_PARAMS):
        axes[1, i].text(0.5, 0.5, "UMAP not installed", ha="center",
                        va="center", transform=axes[1, i].transAxes, fontsize=12)
        axes[1, i].set_title(f"UMAP n_neighbors={nn}", fontsize=11)
        axes[1, i].set_xticks([])
        axes[1, i].set_yticks([])

fig.suptitle("t-SNE vs UMAP Across Parameters (colored by final grade)",
             fontsize=15)
fig.tight_layout()
fig.savefig("q10_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# FIGURE 2 - PARAMETER STUDY: silhouette vs parameter value
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(TSNE_PARAMS, tsne_scores, "o-", color="#2196F3", linewidth=2,
        markersize=8, label="t-SNE (perplexity)")
if UMAP_AVAILABLE:
    ax.plot(UMAP_PARAMS, umap_scores, "s-", color="#FF5722", linewidth=2,
            markersize=8, label="UMAP (n_neighbors)")
ax.set_xlabel("Parameter value")
ax.set_ylabel("Silhouette score (grade separation)")
ax.set_title("Cluster Separation vs Hyper-parameter", fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("q10_parameter_study.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------------
print("\nSILHOUETTE SCORES (higher = grades better separated)")
print("  t-SNE:", dict(zip(TSNE_PARAMS, np.round(tsne_scores, 3))))
if UMAP_AVAILABLE:
    print("  UMAP :", dict(zip(UMAP_PARAMS, np.round(umap_scores, 3))))

print("\nPARAMETER EFFECTS")
print("- t-SNE perplexity balances LOCAL vs GLOBAL structure: small values")
print("  (5) make tiny islands; large values (50) blur groups together.")
print("- UMAP n_neighbors does the same job: small = local detail,")
print("  large = more of the big-picture geometry preserved.")
print("\nt-SNE vs UMAP")
print("- t-SNE: excellent local clusters, but distances BETWEEN clusters and")
print("  their relative sizes are essentially meaningless.")
print("- UMAP: preserves more global structure, usually faster on large data,")
print("  and keeps more of the continuum between groups.")
print("\nLIMITATIONS - read before publishing any embedding")
print("- These maps are for VISUALIZATION only; never cluster or classify on them.")
print("- Different seeds/parameters give different pictures of the SAME data.")
print("- Silhouette above uses grade labels only as a reference for comparison -")
print("  it does NOT prove students naturally form exactly 7 clusters.")
print("- If UMAP is missing: pip install umap-learn  (Kaggle: !pip install umap-learn)")

## 📋 Summary and Key Learnings

1. **Bar / box / histogram** — the three basic views: categories, spread, shape.
2. **Time series** — convert dates first, then group; beware small per-date samples.
3. **Honesty** — zero-based axes, accessible colours, always title + label.
4. **Encoding** — position beats area beats colour (Cleveland–McGill).
5. **EDA** — mean/median/std/skew + IQR outliers tell the distribution story.
6. **Correlation** — read the heatmap, then say it out loud: *correlation ≠ causation*.
7. **Regression** — slope is "marks per hour", R² is "% of variance explained".
8. **Uncertainty** — std band = data spread; CI = precision of the mean.
9. **PCA** — standardize → components → check explained variance before trusting 2D.
10. **t-SNE / UMAP** — pretty pictures, meaningless distances; visualization only.

**Remember:** this dataset is **SYNTHETIC** — generated for learning, not real research.

In [ ]:
# Save everything that was produced
import glob
pngs = sorted(glob.glob("*.png"))
print(f"{len(pngs)} figures saved to {os.getcwd()}/")
for p in pngs:
    print(" -", p)
print("\nAll 10 experiments completed successfully! (10 simple + 10 advanced programs)")